# ema-second-moment composite — cx23: v EMA into the sqrt(v_hat)+eps Adam denominator

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `ema-second-moment`, `sqrt-eps-stabilize`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "ema-second-moment"
DD_ATOM_IDS = ["ema-second-moment", "sqrt-eps-stabilize"]
DD_SUBTOPICS = ["Optimizer: Adam EMA second moment", "Numerical: sqrt-eps stabilization"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

This is the **load-bearing pair** at the heart of Adam's adaptive scale: take the second moment `v` (an EMA of `g**2`), bias-correct to `v_hat`, then form the denominator `sqrt(v_hat) + eps`. The full Adam parameter step divides `m_hat` by this denominator.

**The two atoms.**
- **ema-second-moment** — `v = beta2 * v + (1 - beta2) * g.pow(2)`. Already builds in non-negativity because `g**2 >= 0` and `v` starts at 0.
- **sqrt-eps-stabilize** — `denom = sqrt(v_hat) + eps`. The eps is the guard against early-training `v_hat ~ 0` (e.g. when all gradient entries are zero).

**Why this exact composition.** The whole point of `v` is to be a per-coordinate scale; dividing the parameter step by `sqrt(v_hat)` gives directions with large historical |g| smaller effective steps. Adding `eps` ensures the division is finite when `v_hat` is small.

**Eps placement: OUTSIDE the sqrt (Adam convention).** ```python
denom = (v_hat).sqrt() + eps     # canonical Adam
# NOT: denom = (v_hat + eps).sqrt()  # BatchNorm style — wrong for Adam
```
`torch.optim.Adam` uses outside-sqrt. AdamW uses outside-sqrt. The HuggingFace AdamW uses outside-sqrt. Anything else disagrees with the reference.

**Anatomy.**
```python
def adam_denominator(v, g, beta2, t, eps):
    v = beta2 * v + (1 - beta2) * g.pow(2)            # atom A
    v_hat = v / (1 - beta2 ** t)                       # bias correction (assumed prior atom)
    denom = v_hat.sqrt() + eps                          # atom B (outside-sqrt)
    return v, denom
```

### Composite Exercise — v EMA into the sqrt(v_hat)+eps Adam denominator

**Atoms exercised together**: `ema-second-moment`, `sqrt-eps-stabilize`

Implement `cx23_v_step_and_denom(v, g, beta2, t_step, eps)`.

Inputs:
- `v`: second-moment buffer (Tensor).
- `g`: gradient (Tensor, same shape as `v`).
- `beta2`: float decay (typically 0.999).
- `t_step`: int >= 1.
- `eps`: float stabilizer (typically 1e-8).

Steps:
1. Update `v_new = beta2 * v + (1 - beta2) * g.pow(2)` (atom: ema-second-moment).
2. Bias-correct: `v_hat = v_new / (1 - beta2 ** t_step)`.
3. Build `denom = sqrt(v_hat) + eps` — **eps OUTSIDE the sqrt** (atom: sqrt-eps-stabilize).

Return `(v_new, denom)`. Do not mutate `v`.

Tests verify:
- The EMA recurrence is correct (atom A).
- `denom > 0` strictly, even when `g == 0` and `v == 0`.
- When `g == 0` and `v == 0`, `denom == eps` (proves eps is OUTSIDE the sqrt).
- With constant `g` for many steps, `denom -> |g| + eps` (sqrt of `g**2` recovers `|g|`).
- Cross-check vs `torch.optim.Adam`'s full denominator after one step.

In [ ]:
def cx23_v_step_and_denom(v, g, beta2, t_step, eps):
    # Atom A (ema-second-moment): EMA on g**2.
    v_new = beta2 * v + (1.0 - beta2) * g.pow(2)
    # Bias-correct (intermediate, not its own atom in this drill).
    v_hat = v_new / (1.0 - beta2 ** t_step)
    # Atom B (sqrt-eps-stabilize): Adam convention — eps OUTSIDE the sqrt.
    denom = v_hat.sqrt() + eps
    return v_new, denom


<details><summary>Show solution — cx23</summary>

```python
def cx23_v_step_and_denom(v, g, beta2, t_step, eps):
    # Atom A (ema-second-moment): EMA on g**2.
    v_new = beta2 * v + (1.0 - beta2) * g.pow(2)
    # Bias-correct (intermediate, not its own atom in this drill).
    v_hat = v_new / (1.0 - beta2 ** t_step)
    # Atom B (sqrt-eps-stabilize): Adam convention — eps OUTSIDE the sqrt.
    denom = v_hat.sqrt() + eps
    return v_new, denom
```

**This pair is Adam's actual denominator.** The full step is `p <- p - lr * m_hat / denom`. Getting eps inside vs outside the sqrt is the most common Adam-from-scratch bug after the wrong-beta-divisor bug. Inside-sqrt (BatchNorm style) silently weakens the eps guard — `sqrt(eps) >> eps` when eps is small, so the divide is less aggressive in tiny-`v_hat` regimes.

**The 'constant-g → |g| + eps' invariant** is the cleanest end-to-end check: after many steps, `v_hat = g**2` exactly (by the bias-correction round-trip from cx21), so `sqrt(v_hat) = |g|`, and the denominator carries no information about gradient sign — which is why Adam's per-coordinate scale only cares about magnitude.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx23'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx23',
        'subtopics': ["Optimizer: Adam EMA second moment", "Numerical: sqrt-eps stabilization"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()